# Hidden Markov Models for Regime Prediction

## Introduction

Markets are generalized by their regime, which varies between different magnitudes of being bullish (trending upwards) bearish (trending downwards). Financial analysts may look for patterns in order to determine the most accurate means of predicting upcoming regimes. In order to accurately gauge possible paths, forms of statistical interference and machine learning are regarded as superior. In our case, given an asset, we hope to analyze past drift values in order to construct a means of identifying the probabilities of all possible regimes for the next 5 days (effectively constructing possible "paths" for the asset), given that we know the regime of the present. Our framework of choice is a Hidden Markov Model. 



## HMM Basics and Bayesian Priors

It's necessary to know what a Hidden Markov Model is first. The HMM is comprised mainly of three matrices: the initial matrix, the transition matrix, and the emission matrix. The role of the initial matrix, typically denoted as $\pi$, is to reveal the initial distribution of all states if we are looking at the data purely statistically. 

### $\pi$ Matrix

In our case, we are using past drift values to make our predictions. We are training on the past year of the asset's behavior, so we will look at each daily drift value for the past 252 trading days and place each day in a "bucket" which corresponds to a specific regime. The distribution of the number of days per bucket implicitly reveals the initial distribution of our states, thus allowing us to construct the $\pi$ matrix. Consider the following example:

Say we are analyzing an asset and we calculate the daily drift values using the formula $\mu=\ln(\frac{S_t}{S_{t-1}})$. Now we have a set of 251 $\mu_i$ values. The regimes we wish to identify for our model are **severely bearish, bearish, slightly bearish, stagnant, slightly bullish, bullish,** and **severely bullish**. Their average drift ranges are $(<-0.3), (-0.3, -0.15), (-0.15, -0.05), (-0.05, 0.05), (0.05, 0.15), (0.15, 0.3), (>0.3)$, respectively. Each $\mu_i$ value places perfectly into one bucket. Dividing the number of $\mu_i$ values by 251 normalizes the values, yielding probabilities, thus creating the $\pi$ matrix for our asset.

### Transition Matrix

Next, let's look at how we can construct our transition matrix comprehensively. The transition matrix $A$ reveals the likelihood of another state occurring given a current state. For example (this does not mimic our actual dynamics), we might say that given a bearish regime, we have around a 50% of transitioning to a slightly bearish regime, and around a 5% of transitioning to a stagnant regime. This is done for every state. In our case we have seven states, so our $A$ is a $7\times7$ matrix. Now is a perfect time to introduce the concept of **Bayesian priors**. This is a means of providing some sort of logical framework for our transition and and emission matrices to accelerate the learning process. For example, if our asset consistenty has high volatility, we should skew our distributions in favor of that, since there is a high chance that the asset continues to exhibit high volatility. For our asset, we will use the the long-term variance $\theta$, the volatility-of-volatility $\sigma_v$, and the rate of mean reversion $\kappa$. These parameters are derived from the Heston framework, which is covered in our notebook on volatility surfaces. But, for context:

- Volatility-of-volatility $\sigma_v$:
    - $(<0.30)$ correlates to a slightly bullish trend
    - $(0.30, 0.60)$ is standard volatility behavior
    - $(>0.60)$ correlates to severely bearish or melt-ups
- Mean reversion $\kappa$:
    - $(<2.0)$ correlates to a bearish regime with heavy residuals
    - $(2.0, 5.0)$ is the standard market range
    - $(>5.0)$ correlates to a bullish trend where every dip is bought back up
- Long-term variance $\theta$:
    - $(<0.020)$ correlates to a calm or steadily bullish market
    - $(0.020, 0.040)$ correlates to healthy market consolidation
    - $(0.040, 0.090)$ correlates to bear market trends
    - $(>0.090)$ correlates to crashes and major shocks

Logically, we want our initial transition matrix to mimic these dynamics. We initially set our matrix to relatively logical transition probabilities at a starting point:
```
A_base = np.array([
        [0.70, 0.20, 0.08, 0.02, 0.00, 0.00, 0.00], # 0: Sev Bullish
        [0.10, 0.70, 0.15, 0.05, 0.00, 0.00, 0.00], # 1: Bullish
        [0.02, 0.12, 0.75, 0.09, 0.02, 0.00, 0.00], # 2: Sl Bullish
        [0.00, 0.03, 0.12, 0.70, 0.12, 0.03, 0.00], # 3: Stagnant
        [0.00, 0.00, 0.02, 0.12, 0.70, 0.14, 0.02], # 4: Sl Bearish
        [0.00, 0.00, 0.00, 0.05, 0.15, 0.65, 0.15], # 5: Bearish
        [0.00, 0.00, 0.00, 0.00, 0.05, 0.25, 0.70]  # 6: Sev Bearish
    ])
```
**For $\sigma_v$**, we want to move our values closer to the average, since a higher vol-of-vol correlates to a non-skewed row. We calculate the average of a given row, and then for each probability within the row, we calculate the distance to the average. We then subtract $0.2 \times \operatorname{Distance to Avg.} \times \sigma_v$ from every probability. 

**For $\kappa$**, we want to increase the probability that the current state persists. This involves increasing the diagonals of the matrix each by $0.02 \times \kappa$. 

**For $\theta$**, we want to increase the probability of bearish regimes. For every the bearish probabilities in every row, we scale them in accordance with $\theta$: **severely bearish** $+= 0.5\times\theta$, **bearish** $+=0.25\times\theta$, **slightly bearish** $+=0.125\times\theta$. 

And of course, we normalize every row once more to ensure the sum of each row is capped to one. 

### Emission Matrix

We also use the idea of creating Bayesian priors to update our emission matrix, also using long-term variance $\theta$, the volatility-of-volatility $\sigma_v$, and the rate of mean reversion $\kappa$. We logically create our base emission matrix in accordance with the drift buckets we defined earlier:
```
B_base = np.array([
        [0.85, 0.10, 0.05, 0.00, 0.00, 0.00, 0.00], # 0: (>0.3)
        [0.10, 0.75, 0.10, 0.05, 0.00, 0.00, 0.00], # 1: (0.15, 0.3)
        [0.02, 0.10, 0.76, 0.10, 0.02, 0.00, 0.00], # 2: (0.05, 0.15)
        [0.01, 0.04, 0.15, 0.60, 0.15, 0.04, 0.01], # 3: (-0.05, 0.05)
        [0.00, 0.02, 0.08, 0.15, 0.65, 0.08, 0.02], # 4: (-0.15, -0.05)
        [0.00, 0.00, 0.05, 0.10, 0.15, 0.60, 0.10], # 5: (-0.3, -0.15)
        [0.02, 0.00, 0.00, 0.05, 0.10, 0.23, 0.60]  # 6: (<-0.3) 
    ])
```

Note that the emission matrix is purely quantitative, so we must interpret our Bayesian prior parameters in a purely quantitative/statistical way. And so,

**For $\sigma_v$**, this statistically means our distribution has excess kurtosis or fat tails. So, we increase the tails of each row, the **severely bearish** and **severely bullish** cells, by $0.1\times\sigma_v$.

**For $\kappa$**, the requirement of modifying the diagonal doesn't change. For each cell in the diagonal, increase it by $0.05 \times\kappa$.

**For $\theta$**, this statistically mean our distribution is widened. This is mathematically the same mechanism we implemented for the $\sigma_v$ update for the transition matrix: Take the average of each row, and then per cell in the row, subtract $0.5 \times \operatorname{Distance to Avg.} \times \sigma_v$.

## HMM Learning Process

Now that our matrices are accurately created, we can begin the learning process, which further optimizes the matrices. This process occurs in three steps: the calculation of all $\alpha$ values using the **forward algorithm**, the calculation of all $\beta$ values using the **backwards algorithm**, and the optimization of our matrices using the **Baum-Welch algorithm**. 

### Forward Algorithm

This algorithm is, hence the name, forwards-looking. It produces a matrix of $T$ length and $j$ height (the number of states). Each column effectively represents the probability of viewing each state state at time $t$, when considering the current transition matrix, emission matrix, and all $\alpha$ values in the previous column. To calculate the initial column of the matrix, the formula is:
$$
\alpha_1(i)=\pi_ib_i(O_1)
$$
Where $\pi$ is the initial matrix, $b$ is the observation matrix, and $O_1$ is the initial observation (the first daily drift value). $i$ refers to the current state we are calculating for (this calculation is done for every state). For every subsequent time step, we use this formula:
$$\alpha_t(j)=\Big[\sum_i\alpha_{t-1}(i)a_{ij}\Big]b_j(O_t)$$
Where $j$ refers to the current state. $\alpha(i)_{t-1}$ refers to the alpha value at the previous time step for state $i$ (so, for every $\alpha$ value for a state, we must factor in the $\alpha$ values in the previous column). $a_{ij}$ is the transition matrix, so $P(j|i)$. And again, $b_j$ is the emission matrix, and inputs the daily drift value for day $t$. And remember, a single use of this formula only calculates the $\alpha$ value for a single state at time $t$, so this calculation must be done for every state in order to construct a full column. 

### Backwards Algorithm

This algorithm is backwards-looking. It produces a matrix with the same dimensions as the $\alpha$, but instead calculates the probability of a state appearing given what the next probability was. It effectively serves as a means of providing symmetry for our optimizations. And note that the input order of the $\beta$ matrix is flipped: the column population progression is right to left. Every value in the $\beta_T$ column is set to $1$. For every subsequent column, we use the following formula:
$$\beta_t(i)=\sum_ja_{ij}b_j(O_{t+1})\beta_{t+1}(j)$$
Our variables are the same: we have the transition matrix $a_{ij}$, the emission matrix $b_j$, and our "next" beta values. The only structural difference between the forward algorithm, is that the "next" beta value is incorporated inside the summation. 

### Baum-Welch Algorithm

This process iteratively updates the initial, transition, and emission matrices based off of the actual observed probabilities from the alpha and beta matrices. The first step in this optimization process is to calculate the $\gamma_t(i)$ matrix, which is the join probability of being in state $i$ at time $t$ in the observation sequence:
$$
\gamma_t(i)=\frac{\alpha_t(i)\cdot\beta_t(i)}{\sum_{k=1}^N\alpha_t(k)\beta_t(k)}
$$
Where $N$ is the number of states, and $k$ corresponds to all other states (normalization).

Then we calculate the $\xi_t(i,j)$ matrix, which is the probability of being in state $i$ at time $t$ and state $j$ at time $t+1$:
$$\xi_t(i,j)=\frac{\alpha_t(i)\cdot a_{ij}\cdot b_j(O_{t+1})\cdot\beta_{t+1}(j)}{\sum_{k=1}^N\sum_{w=1}^N\alpha_t(k)a_{kw}b_w(O_{t+1})\beta_{t+1}(w)}$$
This produces a consolidated matrix of the probability of a future state given a current one. Once these two matrices are calculated, we can update our base matrices. 

The **$\pi$ matrix** should be set to the $\gamma_1$ column.

Each value in the **transition matrix** is updated as follows:
$$a_{ij}=\frac{\sum_{t=1}^{T-1}\xi_t(i,j)}{\sum_{t=1}^{T-1}\gamma_t(i)}$$
Which effectively updates the given transition probability $P(i|j)$ based on the actual probability of the given condition $i$.

Each value in the **emission matrix** is updated as follows:
$$
b_{k,j}=\frac{\sum_{t=1}^T\gamma_t(j)\cdot \mathbb{I}(O_t==v_k)}{\sum_{t=1}^T\gamma_t(j)}
$$
Given an observation $k$, we iterate through every state $j$ at all times up to $T$, and zero out any contributions when $O_t$ doesn't match state $k$. This in turn reveals how much time state $j$ spends with observation $k$, thus providing more accurate probabilities. 



## Realistic Smoothing

On its own, the HMM is designed to fit parameters in a purely statistical way. So, we must implement our own smoothing mechanisms in order to effectively smooth the transitions between matrix values. Our first means of doing this involves a small, negligible **pseudocount**, set to $0.0001$. Every value in all updated matrices are incremented by this value, in order to give the almost negligible values additional weighting. We also apply an intertia based parameter update, so that the new contributions do not completely overwrite the previous cell values in all matrices. The formula for this is:
$$\theta^{\operatorname{new}}=\eta\cdot\theta^{old}+(1-\eta)\cdot\theta^{BM}$$





## Execution

For our model, we ran the optimization process 10 times in order to get accurate matrix value. In order to utilize our HMM in order to make regime predictions, we used the following process:
1) Run the forward algorithm to get the probability distribution of all states for tomorrow and normalize each $\alpha_t(i)$ value:
$$\alpha_{t}(i)=\frac{\alpha_{t}(i)}{\sum_{j}\alpha_{t}(j)}$$
2) The formula for determining the distribution for any time $n$ periods ahead is:
$$\alpha_{t+n}=\alpha_{t}A^n$$
Where $A$ is the transition matrix. $\alpha_{t+n}$ is our distribution of states for period $n$.